# Query Transformation Techniques for RAG

## Overview

This notebook is a deep, hands-on exploration of **query transformation** — the family of
techniques that rewrite, expand, decompose, or generalize a user's query *before* it hits
the retriever. In a standard RAG pipeline the retrieval step is the bottleneck: no matter
how good the generator is, if the retrieved chunks are irrelevant the answer will be wrong.
Query transformation attacks this bottleneck from the *query side*.

We implement everything **from scratch** — no LangChain, no LlamaIndex, no OpenAI API.
Just Python, a local LLM, sentence-transformers for embedding, and FAISS for search.

## Techniques Covered

| # | Technique | Core Idea |
|---|---|---|
| 1 | **Multi-Query Generation** | Generate 3-5 reformulations, retrieve for each, fuse with RRF |
| 2 | **Step-Back Prompting** | Abstract the query to a higher conceptual level |
| 3 | **Query Decomposition** | Break a complex query into answerable sub-questions |
| 4 | **Query Expansion** | Add synonyms and related terms to broaden recall |

We test every technique on the same set of queries against a synthetic knowledge base,
compare retrieval quality, discuss when to use each, and build a complete RAG pipeline
with multi-query transformation.

## Part 1 — The Query Problem

### Why Queries Fail

Users ask questions in *natural human language*. Knowledge bases are written in *expository
prose*. The mismatch between these two registers is the single biggest source of retrieval
failure in RAG systems.

**Common failure modes:**

| Failure | Example | What Goes Wrong |
|---|---|---|
| **Vagueness** | "Tell me about transformers" | Model architecture? Electrical? Movie? |
| **Vocabulary mismatch** | "heart attack" vs. "myocardial infarction" | Exact different wording for the same concept |
| **Over-specificity** | "What was GDP growth in Q3 2019?" | If the KB only says "the third quarter showed 2.1% growth" |
| **Multi-hop complexity** | "How did X cause Y which led to Z?" | No single chunk answers the full chain |
| **Implicit context** | "Why did it fail?" | Depends on conversation history |

### The Core Insight

A single query string is a **lossy compression** of the user's actual information need.
Query transformation recovers some of that lost information by generating *multiple*
retrieval probes from different angles. The same information need can be expressed many
ways — and each expression may match different chunks in the vector store.

Think of it like searching a library: instead of walking to one shelf, you send five
librarians to different sections simultaneously, then combine what they bring back.

## Setup

Install dependencies, load the Qwen3-8B model in 4-bit NF4, and define our `generate()` helper.

In [ ]:
# --- Google Colab Setup ---
!pip install -q transformers>=4.51.0 accelerate bitsandbytes torch sentence-transformers faiss-cpu numpy

import torch, json, re, textwrap
import numpy as np
from google.colab import drive
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

drive.mount('/content/drive')
CACHE_DIR = "/content/drive/MyDrive/models"

MODEL_NAME = "Qwen/Qwen3-8B"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, cache_dir=CACHE_DIR)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    quantization_config=quantization_config,
    cache_dir=CACHE_DIR,
    torch_dtype="auto",
)

def generate(messages, max_new_tokens=1024, temperature=0.7):
    """Generate a response from the model given a list of chat messages."""
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, enable_thinking=False
    )
    inputs = tokenizer([text], return_tensors="pt").to(model.device)
    output_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        temperature=temperature,
        do_sample=True,
        top_k=20,
        top_p=0.9,
    )
    generated = output_ids[0][inputs.input_ids.shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True)

print(f"Model loaded: {MODEL_NAME} (4-bit NF4)")
print(f"Device: {model.device}")

## Embedding Model & Vector Store

We use **BGE-base-en-v1.5** from BAAI — a strong open-source embedding model — and
FAISS for approximate nearest-neighbor search.

In [ ]:
from sentence_transformers import SentenceTransformer
import faiss

embed_model = SentenceTransformer("BAAI/bge-base-en-v1.5", cache_folder=CACHE_DIR)

def embed_texts(texts):
    """Embed a list of strings into numpy arrays (768-d)."""
    return embed_model.encode(texts, normalize_embeddings=True, show_progress_bar=False)

def build_faiss_index(chunks):
    """Build a FAISS inner-product index from a list of text chunks."""
    embeddings = embed_texts(chunks)
    dim = embeddings.shape[1]
    index = faiss.IndexFlatIP(dim)          # inner product (cosine with normalized vecs)
    index.add(embeddings.astype(np.float32))
    return index

def search(query, index, chunks, k=5):
    """Return top-k (score, chunk) tuples for a single query string."""
    q_vec = embed_texts([query]).astype(np.float32)
    scores, idxs = index.search(q_vec, k)
    results = []
    for score, idx in zip(scores[0], idxs[0]):
        if idx >= 0:
            results.append((float(score), chunks[idx]))
    return results

print(f"Embedding model loaded: BAAI/bge-base-en-v1.5 (dim={embed_model.get_sentence_embedding_dimension()})")

## Synthetic Knowledge Base

We create a small but carefully designed knowledge base about **renewable energy and climate
policy**. Each chunk is written to test different retrieval challenges: vocabulary mismatch,
specificity, implicit relationships, etc.

In [ ]:
KNOWLEDGE_BASE = [
    # Solar energy
    "Solar photovoltaic (PV) technology converts sunlight directly into electricity using "
    "semiconductor materials. Modern PV panels achieve 20-22% efficiency for residential "
    "installations. The levelized cost of solar energy dropped 89% between 2010 and 2022, "
    "making it the cheapest new source of electricity in most markets worldwide.",

    "Concentrated solar power (CSP) uses mirrors or lenses to focus sunlight onto a receiver "
    "that heats a working fluid. Unlike PV, CSP can store thermal energy in molten salt tanks "
    "for up to 12 hours, enabling electricity generation after sunset. Major CSP plants "
    "operate in Spain, Morocco, and the southwestern United States.",

    # Wind energy
    "Onshore wind turbines typically have capacities of 2-5 MW per unit. The global installed "
    "onshore wind capacity exceeded 800 GW by 2023. Wind energy's main limitation is "
    "intermittency — output varies with weather conditions, requiring grid-level storage "
    "or backup generation for reliability.",

    "Offshore wind farms achieve higher capacity factors (40-50%) compared to onshore (25-35%) "
    "due to stronger, more consistent winds at sea. Floating offshore wind technology, still "
    "in early deployment, could unlock vast deep-water areas previously inaccessible to "
    "fixed-bottom turbines.",

    # Energy storage
    "Lithium-ion batteries dominate grid-scale energy storage with round-trip efficiency of "
    "85-95%. Battery costs fell from $1,200/kWh in 2010 to approximately $140/kWh in 2023. "
    "Alternative technologies include flow batteries for long-duration storage and compressed "
    "air energy storage (CAES) for large-scale applications.",

    "Pumped hydroelectric storage accounts for 95% of installed grid storage capacity "
    "worldwide. It works by pumping water uphill during low-demand periods and releasing "
    "it through turbines during peak demand. New closed-loop designs avoid the environmental "
    "impacts of traditional river-based systems.",

    # Climate policy
    "The Paris Agreement, adopted in 2015, aims to limit global warming to well below 2°C "
    "above pre-industrial levels, with efforts to limit it to 1.5°C. Under the agreement, "
    "countries submit Nationally Determined Contributions (NDCs) outlining their emission "
    "reduction targets and climate action plans.",

    "Carbon pricing mechanisms include cap-and-trade systems and carbon taxes. The EU "
    "Emissions Trading System (EU ETS), the world's largest carbon market, covers about "
    "40% of EU greenhouse gas emissions. Carbon prices in the EU ETS reached over €90 per "
    "tonne of CO2 in 2023, significantly influencing industrial decarbonization decisions.",

    # Grid integration
    "Smart grid technology uses digital communication and automated controls to manage "
    "electricity distribution in real-time. Demand response programs incentivize consumers "
    "to shift electricity usage away from peak periods, reducing the need for expensive "
    "peaker plants and improving grid stability.",

    "Grid interconnection allows regions with surplus renewable energy to export power to "
    "areas with deficits. The European electricity network connects 35 countries, enabling "
    "Norwegian hydropower to balance Danish wind and Spanish solar. High-voltage direct "
    "current (HVDC) transmission lines minimize losses over long distances.",

    # Economics
    "The global renewable energy sector employed approximately 13.7 million people in 2022, "
    "with solar PV being the largest employer at 4.9 million jobs. China, the EU, Brazil, "
    "and the United States are the largest renewable energy employers.",

    "Green hydrogen, produced by electrolyzing water with renewable electricity, could "
    "decarbonize hard-to-abate sectors like steel manufacturing, shipping, and aviation. "
    "Current production costs range from $4-8/kg, but projections suggest costs could fall "
    "below $2/kg by 2030 with scale-up and technological improvements.",

    # Environmental impacts
    "Wind turbines cause bird and bat mortality, with estimates of 140,000-500,000 bird "
    "deaths annually in the United States. Mitigation strategies include radar-activated "
    "shutdown systems, blade painting, and careful site selection to avoid migratory routes.",

    "Large-scale solar farms require significant land area — approximately 5-10 acres per MW. "
    "Agrivoltaics, the co-location of solar panels and agriculture, can reduce land-use "
    "conflicts. Studies show certain crops (lettuce, tomatoes, peppers) actually benefit from "
    "the partial shade provided by elevated solar panels.",

    # Nuclear comparison
    "Nuclear power provides about 10% of global electricity and is the second-largest source "
    "of low-carbon generation after hydropower. Small modular reactors (SMRs), with capacities "
    "under 300 MW, are being developed to offer flexible, factory-built nuclear options that "
    "may be more financially viable than traditional large reactors.",

    # Historical context
    "The oil crises of 1973 and 1979 catalyzed early government investment in renewable "
    "energy research. Denmark's decision to pursue wind power following the 1973 embargo "
    "eventually led to the modern wind industry. Similarly, President Carter installed solar "
    "panels on the White House in 1979, symbolizing early U.S. interest in solar energy.",
]

print(f"Knowledge base: {len(KNOWLEDGE_BASE)} chunks")
for i, chunk in enumerate(KNOWLEDGE_BASE):
    print(f"  [{i:2d}] {chunk[:80]}...")

### Build the FAISS Index

We embed all 16 chunks and build an inner-product index. Since BGE embeddings are
L2-normalized, inner product equals cosine similarity.

In [ ]:
faiss_index = build_faiss_index(KNOWLEDGE_BASE)
print(f"FAISS index built: {faiss_index.ntotal} vectors, dimension {faiss_index.d}")

### Baseline: Single-Query Retrieval

Before applying any transformation, let's see how vanilla single-query retrieval performs.
We define a set of test queries that represent different failure modes.

In [ ]:
TEST_QUERIES = [
    "How can we store energy from the sun when it's not shining?",     # vocabulary mismatch
    "What caused countries to start investing in renewable energy?",    # needs historical context
    "How does renewable energy affect jobs and the economy?",           # broad, multi-faceted
    "What are the environmental downsides of clean energy?",            # counter-intuitive
    "How do countries work together on climate goals?",                 # requires policy + grid knowledge
]

def print_results(results, label="Results"):
    """Pretty-print retrieval results."""
    print(f"\n{'─'*80}")
    print(f" {label}")
    print(f"{'─'*80}")
    for rank, (score, chunk) in enumerate(results, 1):
        short = chunk[:100].replace('\n', ' ')
        print(f"  #{rank}  (score={score:.4f})  {short}...")

# Baseline retrieval for each test query
print("=" * 80)
print(" BASELINE: Single-Query Retrieval (top-3)")
print("=" * 80)
for q in TEST_QUERIES:
    print(f"\n▶ Query: {q}")
    results = search(q, faiss_index, KNOWLEDGE_BASE, k=3)
    print_results(results, "Top-3 baseline")

---

## Part 2 — Multi-Query Generation

### Concept

The same information need can be expressed many different ways. Multi-query generation
uses the LLM to produce **3-5 reformulations** of the original query, each phrasing the
question from a slightly different angle or using different vocabulary.

We then retrieve for *each* reformulation independently and **fuse** the result lists
using **Reciprocal Rank Fusion (RRF)** — a rank-based merging algorithm that doesn't
require comparable scores across queries.

### Why It Works

- **Vocabulary coverage**: Different phrasings match different chunks.
- **Angle diversity**: Each reformulation may emphasize a different aspect.
- **Robustness**: If one reformulation fails, others may succeed.

### RRF Formula

For a document $d$ appearing at rank $r_i$ in result list $i$:

$$\text{RRF}(d) = \sum_{i=1}^{n} \frac{1}{k + r_i}$$

where $k$ is a smoothing constant (typically 60). Documents not in a result list get no
contribution from that list.

In [ ]:
def generate_multi_queries(original_query, n_queries=4):
    """Use the LLM to generate multiple reformulations of a query."""
    messages = [
        {"role": "system", "content": (
            "You are a helpful assistant that generates alternative search queries. "
            "Given a user question, generate alternative versions that approach the "
            "same information need from different angles, using different vocabulary. "
            f"Return exactly {n_queries} queries, one per line, numbered 1-{n_queries}. "
            "Do NOT include the original query. Do NOT add explanations."
        )},
        {"role": "user", "content": f"Original query: {original_query}"}
    ]
    response = generate(messages, max_new_tokens=300, temperature=0.7)

    # Parse numbered lines
    queries = []
    for line in response.strip().split("\n"):
        line = line.strip()
        cleaned = re.sub(r'^\d+[\.\)\-]\s*', '', line).strip()
        if cleaned and len(cleaned) > 10:
            queries.append(cleaned)
    return queries[:n_queries]


# Demonstrate multi-query generation
demo_query = TEST_QUERIES[0]  # "How can we store energy from the sun..."
print(f"Original query: {demo_query}\n")
alt_queries = generate_multi_queries(demo_query)
print(f"Generated {len(alt_queries)} alternative queries:")
for i, q in enumerate(alt_queries, 1):
    print(f"  {i}. {q}")

### Reciprocal Rank Fusion (RRF)

RRF is elegant because it only depends on **rank position**, not raw scores. This makes
it safe to merge result lists from different queries (which may have incomparable score
distributions). A document that consistently appears near the top across multiple queries
will dominate.

In [ ]:
def reciprocal_rank_fusion(result_lists, k=60):
    """Fuse multiple ranked result lists using RRF.

    Args:
        result_lists: list of lists, each containing (score, chunk) tuples
        k: smoothing constant (default 60)

    Returns:
        Sorted list of (rrf_score, chunk) tuples
    """
    rrf_scores = {}  # chunk -> cumulative RRF score
    for results in result_lists:
        for rank, (score, chunk) in enumerate(results, 1):
            if chunk not in rrf_scores:
                rrf_scores[chunk] = 0.0
            rrf_scores[chunk] += 1.0 / (k + rank)

    fused = sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)
    return [(score, chunk) for chunk, score in fused]


def multi_query_retrieve(original_query, index, chunks, n_queries=4, k=5):
    """Full multi-query retrieval pipeline: generate variants, retrieve, fuse."""
    alt_queries = generate_multi_queries(original_query, n_queries)
    all_queries = [original_query] + alt_queries

    result_lists = []
    for q in all_queries:
        results = search(q, index, chunks, k=k)
        result_lists.append(results)

    fused = reciprocal_rank_fusion(result_lists)
    return fused, all_queries


# Demo: multi-query retrieval on the first test query
fused_results, all_q = multi_query_retrieve(demo_query, faiss_index, KNOWLEDGE_BASE)
print(f"Queries used ({len(all_q)} total):")
for i, q in enumerate(all_q):
    tag = "(original)" if i == 0 else "(generated)"
    print(f"  {i+1}. {tag} {q}")
print_results(fused_results[:5], "Multi-Query + RRF Fusion (top-5)")

---

## Part 3 — Step-Back Prompting

### Concept

Step-back prompting generalizes the query to a **higher conceptual level**. Instead of
searching for the specific question, we also search for the broader category it belongs to.

**Example:**
- Original: *"What caused the 2008 financial crisis?"*
- Step-back: *"What are common causes of financial crises?"*

The step-back query retrieves foundational knowledge that provides context for answering
the specific question. We then perform **dual retrieval**: both the original and the
step-back query hit the index, and we merge the results.

### When to Use

- Queries that reference very specific events, dates, or entities
- "Why" and "How" questions that benefit from background context
- Questions where the user assumes domain knowledge

In [ ]:
def generate_step_back_query(original_query):
    """Use the LLM to generate a more general, abstracted version of the query."""
    messages = [
        {"role": "system", "content": (
            "You are a helpful assistant that generates step-back questions. "
            "Given a specific question, generate a single more general question that "
            "abstracts away the specifics. The general question should capture the "
            "broader category or principle behind the specific question.\n"
            "Return ONLY the step-back question, nothing else."
        )},
        {"role": "user", "content": f"Specific question: {original_query}"}
    ]
    response = generate(messages, max_new_tokens=150, temperature=0.5)
    # Take the first non-empty line
    for line in response.strip().split("\n"):
        line = line.strip()
        if line and len(line) > 10:
            return re.sub(r'^\d+[\.\)\-]\s*', '', line).strip()
    return response.strip()


def step_back_retrieve(original_query, index, chunks, k=5):
    """Dual retrieval: original query + step-back query, merged via RRF."""
    step_back = generate_step_back_query(original_query)
    original_results = search(original_query, index, chunks, k=k)
    step_back_results = search(step_back, index, chunks, k=k)
    fused = reciprocal_rank_fusion([original_results, step_back_results])
    return fused, step_back


# Demo: step-back on each test query
for q in TEST_QUERIES:
    fused, sb = step_back_retrieve(q, faiss_index, KNOWLEDGE_BASE, k=3)
    print(f"\n▶ Original:   {q}")
    print(f"  Step-back:  {sb}")
    print(f"  Top result: {fused[0][1][:90]}...")

### How Step-Back Helps: A Concrete Example

Consider the query *"What caused countries to start investing in renewable energy?"*

The **baseline** retriever might match chunks about current economics (jobs, costs) because
they mention "renewable energy" prominently. But the **historical chunk** about the 1973 oil
crisis uses different vocabulary ("oil crises", "catalyzed", "early government investment").

The step-back query — something like *"What are the historical drivers of energy policy changes?"*
— is more likely to surface that historical chunk because it abstracts to the level where
cause-and-effect language matches.

In [ ]:
# Full step-back comparison for historical query
hist_query = TEST_QUERIES[1]  # "What caused countries to start investing..."

print("BASELINE (single query, top-5):")
baseline = search(hist_query, faiss_index, KNOWLEDGE_BASE, k=5)
print_results(baseline, "Baseline")

print("\nSTEP-BACK RETRIEVAL (top-5):")
sb_results, sb_query = step_back_retrieve(hist_query, faiss_index, KNOWLEDGE_BASE, k=5)
print(f"  Step-back query: {sb_query}")
print_results(sb_results[:5], "Step-Back + RRF")

---

## Part 4 — Query Decomposition

### Concept

Some questions are inherently **multi-hop** — answering them requires combining information
from multiple sources. Query decomposition breaks a complex question into **2-4 simpler
sub-questions**, retrieves for each independently, then synthesizes the sub-answers.

**Example:**
- Original: *"How does renewable energy affect jobs and the economy?"*
- Sub-questions:
  1. *"How many people are employed in the renewable energy sector?"*
  2. *"What is the economic cost comparison of renewable vs. fossil energy?"*
  3. *"What economic opportunities does green hydrogen create?"*

### Why It Works

Each sub-question is **narrower and more specific** than the original, which means:
- Better vocabulary alignment with individual chunks
- Higher retrieval precision per sub-question
- The final synthesis covers multiple facets of the original question

### When to Use

- Complex "how" and "what" questions that span multiple topics
- Comparative questions ("X vs. Y")
- Questions with multiple implicit sub-parts

In [ ]:
def decompose_query(original_query, max_subqueries=4):
    """Use the LLM to break a complex query into simpler sub-questions."""
    messages = [
        {"role": "system", "content": (
            "You are a helpful assistant that breaks complex questions into simpler "
            "sub-questions. Given a complex query, decompose it into 2-4 simpler, "
            "self-contained sub-questions that together would provide a comprehensive "
            "answer to the original.\n"
            f"Return exactly 2-{max_subqueries} sub-questions, one per line, numbered. "
            "Do NOT include explanations."
        )},
        {"role": "user", "content": f"Complex query: {original_query}"}
    ]
    response = generate(messages, max_new_tokens=300, temperature=0.5)

    sub_queries = []
    for line in response.strip().split("\n"):
        cleaned = re.sub(r'^\d+[\.\)\-]\s*', '', line.strip()).strip()
        if cleaned and len(cleaned) > 10:
            sub_queries.append(cleaned)
    return sub_queries[:max_subqueries]


def decomposition_retrieve(original_query, index, chunks, k=3):
    """Decompose, retrieve per sub-question, fuse results."""
    sub_queries = decompose_query(original_query)
    result_lists = []
    for sq in sub_queries:
        results = search(sq, index, chunks, k=k)
        result_lists.append(results)
    fused = reciprocal_rank_fusion(result_lists)
    return fused, sub_queries


# Demo: decompose the broad economic query
econ_query = TEST_QUERIES[2]  # "How does renewable energy affect jobs and the economy?"
print(f"Original: {econ_query}\n")
sub_qs = decompose_query(econ_query)
print("Sub-questions:")
for i, sq in enumerate(sub_qs, 1):
    print(f"  {i}. {sq}")

In [ ]:
# Full decomposition retrieval
dec_results, dec_subqs = decomposition_retrieve(econ_query, faiss_index, KNOWLEDGE_BASE, k=3)

print(f"Query: {econ_query}")
print(f"\nDecomposed into {len(dec_subqs)} sub-questions:")
for i, sq in enumerate(dec_subqs, 1):
    print(f"  {i}. {sq}")

print_results(dec_results[:5], "Decomposition + RRF Fusion (top-5)")

print("\nBaseline comparison (single query, top-5):")
baseline = search(econ_query, faiss_index, KNOWLEDGE_BASE, k=5)
print_results(baseline, "Baseline")

---

## Part 5 — Query Expansion

### Concept

Query expansion adds **synonyms, related terms, and contextual keywords** to the original
query. Unlike multi-query (which creates entirely new questions), expansion augments the
*same* query with additional terms to broaden the retrieval surface.

**Example:**
- Original: *"machine learning"*
- Expanded: *"machine learning artificial intelligence deep learning neural networks statistical learning"*

### Why It Works

Embedding models are good at semantic similarity but not perfect — adding explicit synonyms
and related terms increases the chance that the embedding vector overlaps with relevant
chunks' vectors. This is especially valuable when:

- Domain-specific jargon varies between the query and the knowledge base
- The embedding model wasn't fine-tuned on your specific domain
- The query uses colloquial language but the KB uses technical terms

In [ ]:
def expand_query(original_query):
    """Use the LLM to add related terms and synonyms to the query."""
    messages = [
        {"role": "system", "content": (
            "You are a helpful assistant that expands search queries with related terms.\n"
            "Given a query, add synonyms, related technical terms, and contextual keywords "
            "that would help retrieve relevant documents.\n"
            "Return a single expanded query (one line) that includes the original terms "
            "plus 5-8 additional related terms. Do NOT add explanations.\n"
            "Example:\n"
            "  Input: 'heart attack treatment'\n"
            "  Output: 'heart attack treatment myocardial infarction cardiac emergency "
            "stent angioplasty thrombolysis cardiovascular therapy'"
        )},
        {"role": "user", "content": f"Query: {original_query}"}
    ]
    response = generate(messages, max_new_tokens=200, temperature=0.5)
    # Take the first substantive line
    for line in response.strip().split("\n"):
        line = line.strip()
        if line and len(line) > 15:
            # Remove any leading label like "Output:" or "Expanded:"
            cleaned = re.sub(r'^(output|expanded|expanded query)[:\s]*', '', line, flags=re.IGNORECASE).strip()
            return cleaned
    return response.strip()


def expansion_retrieve(original_query, index, chunks, k=5):
    """Retrieve using the expanded query."""
    expanded = expand_query(original_query)
    results = search(expanded, index, chunks, k=k)
    return results, expanded


# Demo: expand each test query
for q in TEST_QUERIES:
    expanded = expand_query(q)
    print(f"\n▶ Original:  {q}")
    print(f"  Expanded:  {expanded}")

In [ ]:
# Full expansion retrieval comparison
env_query = TEST_QUERIES[3]  # "What are the environmental downsides of clean energy?"

print(f"Query: {env_query}\n")

print("BASELINE (top-5):")
baseline = search(env_query, faiss_index, KNOWLEDGE_BASE, k=5)
print_results(baseline, "Baseline")

print("\nEXPANSION (top-5):")
exp_results, exp_query = expansion_retrieve(env_query, faiss_index, KNOWLEDGE_BASE, k=5)
print(f"  Expanded query: {exp_query}")
print_results(exp_results, "Expanded")

---

## Part 6 — Comparison of All Techniques

Now we run all four techniques (plus baseline) on every test query and compare the
top-5 results. We focus on two metrics:

1. **Unique chunks surfaced**: How many distinct chunks appear in the top-5 across
   techniques? More diversity = better coverage.
2. **Rank of the "ideal" chunk**: For each query, we identify the most relevant chunk
   manually and check where it appears in each technique's ranking.

In [ ]:
# Ideal chunk indices for each test query (manually identified)
IDEAL_CHUNKS = {
    TEST_QUERIES[0]: [1, 4, 5],   # CSP thermal storage, batteries, pumped hydro
    TEST_QUERIES[1]: [15],        # Oil crises / historical context
    TEST_QUERIES[2]: [10, 11],    # Jobs + green hydrogen economics
    TEST_QUERIES[3]: [12, 13],    # Bird mortality + land use
    TEST_QUERIES[4]: [6, 9],      # Paris Agreement + grid interconnection
}

def find_rank(results, target_chunks, knowledge_base):
    """Find the best rank at which any ideal chunk appears."""
    for rank, (score, chunk) in enumerate(results, 1):
        idx = knowledge_base.index(chunk)
        if idx in target_chunks:
            return rank
    return None  # not found in results

def run_all_techniques(query, index, chunks, k=5):
    """Run all 4 techniques + baseline and return results dict."""
    baseline = search(query, index, chunks, k=k)
    mq_results, _ = multi_query_retrieve(query, index, chunks, n_queries=4, k=k)
    sb_results, _ = step_back_retrieve(query, index, chunks, k=k)
    dec_results, _ = decomposition_retrieve(query, index, chunks, k=k)
    exp_results, _ = expansion_retrieve(query, index, chunks, k=k)
    return {
        "Baseline": baseline[:k],
        "Multi-Query": mq_results[:k],
        "Step-Back": sb_results[:k],
        "Decomposition": dec_results[:k],
        "Expansion": exp_results[:k],
    }

# Run comparison on all test queries
print(f"{'Query':<55} {'Technique':<16} {'Ideal Rank':>10} {'Unique':>8}")
print("=" * 95)

for query in TEST_QUERIES:
    ideals = IDEAL_CHUNKS[query]
    all_results = run_all_techniques(query, faiss_index, KNOWLEDGE_BASE, k=5)

    for tech_name, results in all_results.items():
        rank = find_rank(results, ideals, KNOWLEDGE_BASE)
        rank_str = f"#{rank}" if rank else "not found"
        unique_chunks = len(set(c for _, c in results))
        q_short = query[:53] + ".." if len(query) > 53 else query
        print(f"{q_short:<55} {tech_name:<16} {rank_str:>10} {unique_chunks:>8}")
    print("-" * 95)

### Analysis: When to Use Which Technique

| Technique | Best For | Cost | Latency |
|---|---|---|---|
| **Multi-Query** | Vocabulary mismatch, ambiguous queries | 1 LLM call + N retrievals | Medium |
| **Step-Back** | Specific questions needing context | 1 LLM call + 2 retrievals | Low |
| **Decomposition** | Complex multi-part questions | 1 LLM call + N retrievals + synthesis | High |
| **Expansion** | Domain jargon, colloquial queries | 1 LLM call + 1 retrieval | Low |

**Key insights:**

- **Multi-Query** is the most general-purpose technique — it works well across most failure modes.
- **Step-Back** excels when the query is *too specific* and needs broader context.
- **Decomposition** shines for genuinely complex questions but adds the most latency.
- **Expansion** is the lightest-weight option and works well for vocabulary mismatch,
  but doesn't help with structural query issues.

---

## Part 7 — Combining Techniques

Techniques can be **layered**. Common combinations:

1. **Step-Back + Multi-Query**: Generate step-back query, then multi-query *both* the
   original and the step-back. Maximum coverage.
2. **Decomposition + Expansion**: Decompose into sub-queries, expand each before retrieval.
3. **Expansion + Multi-Query**: Expand the original, then generate variants of the expanded query.

The general principle: **start with the technique that addresses the primary failure mode**,
then layer a second technique if needed. Avoid layering more than two — the added LLM calls
increase latency and can introduce noise.

In [ ]:
def combined_stepback_multiquery(original_query, index, chunks, k=5):
    """Combine step-back + multi-query for maximum coverage."""
    # Step 1: Generate step-back query
    step_back = generate_step_back_query(original_query)

    # Step 2: Multi-query both original and step-back
    orig_variants = generate_multi_queries(original_query, n_queries=3)
    sb_variants = generate_multi_queries(step_back, n_queries=2)

    all_queries = [original_query, step_back] + orig_variants + sb_variants

    # Step 3: Retrieve for all and fuse
    result_lists = [search(q, index, chunks, k=k) for q in all_queries]
    fused = reciprocal_rank_fusion(result_lists)

    return fused, all_queries


# Demo: combined approach on the hardest query
hard_query = TEST_QUERIES[4]  # "How do countries work together on climate goals?"

print(f"Query: {hard_query}\n")

combined_results, combined_queries = combined_stepback_multiquery(
    hard_query, faiss_index, KNOWLEDGE_BASE, k=5
)

print(f"Total queries generated: {len(combined_queries)}")
for i, q in enumerate(combined_queries):
    tag = "original" if i == 0 else ("step-back" if i == 1 else "variant")
    print(f"  [{tag}] {q}")

print_results(combined_results[:5], "Combined Step-Back + Multi-Query (top-5)")

# Compare with baseline
print("\nBaseline comparison:")
baseline = search(hard_query, faiss_index, KNOWLEDGE_BASE, k=5)
print_results(baseline, "Baseline (top-5)")

In [ ]:
def combined_decompose_expand(original_query, index, chunks, k=5):
    """Decompose into sub-questions, expand each, retrieve, fuse."""
    sub_queries = decompose_query(original_query)

    result_lists = []
    expanded_queries = []
    for sq in sub_queries:
        expanded = expand_query(sq)
        expanded_queries.append(expanded)
        results = search(expanded, index, chunks, k=k)
        result_lists.append(results)

    fused = reciprocal_rank_fusion(result_lists)
    return fused, sub_queries, expanded_queries


# Demo on broad query
broad_query = TEST_QUERIES[2]  # "How does renewable energy affect jobs and the economy?"
print(f"Query: {broad_query}\n")

dec_exp_results, subs, expanded = combined_decompose_expand(
    broad_query, faiss_index, KNOWLEDGE_BASE, k=5
)

print("Sub-questions → Expanded:")
for sq, eq in zip(subs, expanded):
    print(f"  Q: {sq}")
    print(f"  E: {eq[:100]}...\n")

print_results(dec_exp_results[:5], "Decompose + Expand (top-5)")

---

## Part 8 — Complete RAG Pipeline with Multi-Query Transformation

Now we put it all together: a complete RAG pipeline that uses multi-query transformation
as the default query processing step. The pipeline:

1. Takes a user question
2. Generates query variants via multi-query generation
3. Retrieves for each variant from the FAISS index
4. Fuses results with RRF
5. Formats the top-k chunks as context
6. Sends context + question to the LLM for answer generation

This is a production-ready pattern (minus scaling concerns) that you can adapt to any
knowledge base.

In [ ]:
class QueryTransformRAGPipeline:
    """Complete RAG pipeline with pluggable query transformation."""

    def __init__(self, chunks, index, transform_fn=None, top_k=5):
        self.chunks = chunks
        self.index = index
        self.transform_fn = transform_fn or self._default_transform
        self.top_k = top_k

    def _default_transform(self, query):
        """Default: multi-query with RRF fusion."""
        fused, queries = multi_query_retrieve(query, self.index, self.chunks, k=self.top_k)
        return fused[:self.top_k], queries

    def retrieve(self, query):
        """Retrieve relevant chunks using the configured transformation."""
        results, metadata = self.transform_fn(query)
        return results[:self.top_k], metadata

    def format_context(self, results):
        """Format retrieved chunks into a context string."""
        context_parts = []
        for i, (score, chunk) in enumerate(results, 1):
            context_parts.append(f"[Source {i}] {chunk}")
        return "\n\n".join(context_parts)

    def answer(self, query):
        """Full pipeline: transform → retrieve → generate answer."""
        results, metadata = self.retrieve(query)
        context = self.format_context(results)

        messages = [
            {"role": "system", "content": (
                "You are a knowledgeable assistant. Answer the user's question based "
                "ONLY on the provided context. If the context doesn't contain enough "
                "information, say so. Cite source numbers [Source N] in your answer."
            )},
            {"role": "user", "content": (
                f"Context:\n{context}\n\nQuestion: {query}"
            )}
        ]
        answer = generate(messages, max_new_tokens=512, temperature=0.3)
        return {
            "query": query,
            "answer": answer,
            "sources": results,
            "metadata": metadata,
        }

# Create the pipeline
rag = QueryTransformRAGPipeline(KNOWLEDGE_BASE, faiss_index, top_k=5)
print("RAG pipeline initialized with multi-query transformation.")
print(f"  Knowledge base: {len(KNOWLEDGE_BASE)} chunks")
print(f"  Index: {faiss_index.ntotal} vectors")
print(f"  Top-k: {rag.top_k}")

### Running the Full Pipeline

Let's test the complete pipeline on all our test queries and see the generated answers.

In [ ]:
for q in TEST_QUERIES:
    result = rag.answer(q)

    print(f"\n{'═'*80}")
    print(f"Question: {result['query']}")
    print(f"{'─'*80}")
    print(f"Queries used: {len(result['metadata'])}")
    for i, mq in enumerate(result['metadata']):
        tag = '(original)' if i == 0 else '(variant) '
        print(f"  {tag} {mq}")
    print(f"{'─'*80}")
    print(f"Answer:\n{result['answer']}")
    print(f"{'─'*80}")
    print(f"Sources retrieved: {len(result['sources'])}")
    for i, (score, chunk) in enumerate(result['sources'], 1):
        print(f"  [{i}] (score={score:.4f}) {chunk[:80]}...")

### Pluggable Transforms

The pipeline accepts any transformation function. Let's swap in step-back and decomposition.

In [ ]:
# Step-back pipeline
def stepback_transform(query):
    fused, sb = step_back_retrieve(query, faiss_index, KNOWLEDGE_BASE, k=5)
    return fused[:5], [query, sb]

rag_stepback = QueryTransformRAGPipeline(
    KNOWLEDGE_BASE, faiss_index, transform_fn=stepback_transform, top_k=5
)

# Test on historical query
result = rag_stepback.answer(TEST_QUERIES[1])
print(f"Question: {result['query']}")
print(f"Step-back query: {result['metadata'][1]}")
print(f"\nAnswer:\n{result['answer']}")

# Decomposition pipeline
print("\n" + "═" * 80)

def decompose_transform(query):
    fused, subs = decomposition_retrieve(query, faiss_index, KNOWLEDGE_BASE, k=5)
    return fused[:5], subs

rag_decompose = QueryTransformRAGPipeline(
    KNOWLEDGE_BASE, faiss_index, transform_fn=decompose_transform, top_k=5
)

result = rag_decompose.answer(TEST_QUERIES[2])
print(f"\nQuestion: {result['query']}")
print("Sub-questions:")
for i, sq in enumerate(result['metadata'], 1):
    print(f"  {i}. {sq}")
print(f"\nAnswer:\n{result['answer']}")

---

## Summary

Query transformation is one of the highest-impact improvements you can make to a RAG
system, often more impactful than changing the embedding model or the chunking strategy.

### What We Built

| Component | Implementation |
|---|---|
| **Multi-Query Generation** | LLM generates 3-5 reformulations → RRF fusion |
| **Step-Back Prompting** | LLM abstracts to higher level → dual retrieval |
| **Query Decomposition** | LLM breaks into sub-questions → per-question retrieval → fusion |
| **Query Expansion** | LLM adds synonyms/related terms → single expanded retrieval |
| **Combined approaches** | Step-back + multi-query, decomposition + expansion |
| **Complete RAG pipeline** | Pluggable transformation → FAISS retrieval → LLM generation |

### Key Takeaways

1. **A single query is a lossy representation** of the user's information need.
2. **Multi-query is the best default** — general-purpose, moderate cost, high impact.
3. **Step-back helps when the query is too specific** and needs broader context.
4. **Decomposition handles multi-hop questions** but adds latency.
5. **Expansion is lightweight** and addresses vocabulary mismatch.
6. **RRF is the glue** — it safely merges results from different queries without needing
   comparable scores.
7. **Techniques can be layered**, but rarely more than two at a time.

### Production Considerations

- **Cache transformed queries** to avoid redundant LLM calls for repeated questions.
- **Set timeouts** on LLM query generation — if it's slow, fall back to the original query.
- **Monitor which technique improves results** per query type (use A/B testing).
- **Consider hybrid approaches**: use a lightweight classifier to pick the best technique
  per query instead of always using the same one.